### this script prepare the table for regression analysis

In [36]:
import pandas as pd

In [37]:
pub_author_path='../../data/dropbox/LaborDivision/data/deriveddata/intl_noalpha/pub_author.csv'
author_info_path='../../data/dropbox/LaborDivision/data/cleanedraw/author_info.pkl'
stgroup_path = '../../data/dropbox/LaborDivision/data/additionaldata/cntry_classification/cntry_group_meta.csv'
pub_funding_path = "../../data/dropbox/LaborDivision/data/deriveddata/all/pub_funding.pkl"
cntry_region="../../data/dropbox/LaborDivision/data/additionaldata/cntry_region.xlsx"
cntry_meta="../../data/dropbox/LaborDivision/data/additionaldata/cntry_meta.csv"

In [87]:
regression_table_path = "../../data/dropbox/LaborDivision/data/deriveddata/intl_noalpha/regression_table.csv"

In [3]:
pub_author = pd.read_csv(pub_author_path)
pub_author.head()

,ost_bk,cluster_id,cntry,role,year,dis
0,29768070,65683243.0,JPN,F,2008,51
1,29768070,79129017.0,JPN,M,2008,51
2,29768070,58951927.0,CHN,M,2008,51
3,29768070,9735544.0,JPN,M,2008,51
4,29768070,34120990.0,JPN,L,2008,51


In [4]:
author_df=pd.read_pickle(author_info_path)
author_df.head()

,cluster_id,gender,yfp,npub
0,81178534.0,F,2012.0,11.0
1,4745329.0,F,2013.0,746.0
2,30161693.0,U,2013.0,1.0
3,88125653.0,M,2003.0,111.0
4,22288059.0,M,2000.0,258.0


### merge author information with paper information

In [63]:
reg_df = pub_author.merge(author_df, on='cluster_id')

In [64]:
len(reg_df.ost_bk.unique())

3195989

In [40]:
reg_df.head()

,ost_bk,cluster_id,cntry,role,year,dis,gender,yfp,npub
0,29768070,65683243.0,JPN,F,2008,51,F,2008.0,5.0
1,31240446,65683243.0,JPN,F,2008,51,F,2008.0,5.0
2,29768070,79129017.0,JPN,M,2008,51,M,2000.0,98.0
3,31240446,79129017.0,JPN,M,2008,51,M,2000.0,98.0
4,39595623,79129017.0,JPN,F,2013,18,M,2000.0,98.0


### add country scientific capacity information

In [65]:
group_df=pd.read_csv(stgroup_path)
group_df=group_df[['Code','ST']].drop_duplicates()
group_df = group_df.replace(to_replace={'ST':{'Others':'Lagging'}})

In [67]:
group_df

,Code,ST
0,AFG,Lagging
1,ALB,Lagging
2,DZA,Lagging
3,AND,Lagging
4,AGO,Lagging
...,...,...
197,VEN,Developing
198,VNM,Lagging
199,YEM,Lagging
200,ZMB,Lagging


In [68]:
reg_df = reg_df.merge(group_df, left_on='cntry', right_on='Code')

In [69]:
reg_df.head()

,ost_bk,cluster_id,cntry,role,year,dis,gender,yfp,npub,Code,ST
0,29768070,65683243.0,JPN,F,2008,51,F,2008.0,5.0,JPN,Advanced
1,31240446,65683243.0,JPN,F,2008,51,F,2008.0,5.0,JPN,Advanced
2,29768070,79129017.0,JPN,M,2008,51,M,2000.0,98.0,JPN,Advanced
3,31240446,79129017.0,JPN,M,2008,51,M,2000.0,98.0,JPN,Advanced
4,39595623,79129017.0,JPN,F,2013,18,M,2000.0,98.0,JPN,Advanced


In [70]:
len(reg_df.ost_bk.unique())

3195989

### add funding information to the data

In [39]:
pub_fund = pd.read_pickle(pub_funding_path)
pub_fund.head()

,ost_bk,funders
0,29768167,[GBR]
1,29768198,[IRN]
2,29768203,[USA]
3,29768231,[USA]
4,29768270,[USA]


In [71]:
reg_df = reg_df.merge(pub_fund, on='ost_bk', how='left')
reg_df['funders'] = reg_df['funders'].fillna('nofunding')

In [75]:
#get list of countries that are eu country
region=pd.read_excel(cntry_region)[['cntry','eulabel']]
eu=region[region.eulabel==1]
cntry_code=pd.read_csv(cntry_meta)[['cntry','Code']]
eu=eu.merge(cntry_code,on='cntry',how='left')
eulist=eu.Code.tolist()

In [80]:
def determine_fund(author_cntry, funder_cntry):
    if funder_cntry in ['nofunding','incomplete']:
        return 'invalid'
    elif author_cntry in funder_cntry:
        return 1
    elif 'EU' in funder_cntry and author_cntry in eulist:
        return 1
    else:
        return 0
reg_df['funded'] = reg_df.apply(lambda x:determine_fund(x.cntry, x.funders), axis=1)

In [88]:
reg_df.drop(columns=['Code','funders']).to_csv(regression_table_path, index=False)

In [92]:
reg_df.shape

(20930855, 13)

In [90]:
test = pd.read_csv('paper_fixed_data.csv')

In [91]:
test.shape

(14883457, 14)